# Extended Kalman Filter (EKF)

Intuition: IMU data is used for prediction on each step, visual measurement data is used for periodical corrections.

## Model and notation

An EKF estimates a nonlinear system's **state** $x$ and **state error covariance** $P$. Uncertainty propagation and correction use local Jacobians.

$$
\begin{gather*}
\mathbf{x}_k=f(\mathbf{x}_{k-1},\mathbf{u}_k)+\mathbf{w}_k,
\\
z_k=h(\mathbf{x}_k)+\boldsymbol{\nu}_k
\end{gather*}
$$

Variables:
- $x$ – system state estimated.
- $z$ – actual sensor measurement (camera feature measurement).
- $u$ – input/control (IMU readings).
- $w$ - process noise.
- $v$ - measurement noise.

Functions:
- $f$ - state transition function;
- $h$ - state-to-measurement space mapping function.

Matrices:
- $P$ - state estimate error covariance matrix.
- $Q$ - zero-mean process noise covariance matrix.
- $R$ - zero-mean measurement noise covariance matrix.
- $F$ - Jacobian of $f$ w.r.t. $x$. $F_k=\left.\frac{\partial f}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_{k-1}^+,\mathbf{u}_k}$
- $H$ - Jacobian of $h$ w.r.t. $x$. $H_k=\left.\frac{\partial h}{\partial\mathbf{x}}\right|_{\hat{\mathbf{x}}_k^-}$
- $K$ - Kalman gain determines how strongly each residual affects the state. It depends on both the predicted uncertainty and the uncertainty of the measurement.

Note:
- Superscripts $-$ and $+$ denote estimates **before** and **after** the measurement update.
- Matrices size for state dimension $n$ and measurement dimension $m$: $P,F,Q\in\mathbb{R}^{n\times n}$, $H\in\mathbb{R}^{m\times n}$, $R\in\mathbb{R}^{m\times m}$, $K\in\mathbb{R}^{n\times m}$.

## 1. Predict

Propagate the last corrected state using the new input, then propagate its covariance:

$$
\hat{\mathbf{x}}_k^-=f(\hat{\mathbf{x}}_{k-1}^+,\mathbf{u}_k),
$$

$$
P_k^-=F_kP_{k-1}^+F_k^T+Q_k
$$

Repeat prediction whenever a new motion input/control arrives (new IMU data);

A measurement is not required for this step (visual data).

## 2. Update from a measurement

1. Computer residuals (compare the actual measurement against what the predicted state says the sensor should observe):

   $$
   \mathbf{r}_k=\mathbf{z}_k-h(\hat{\mathbf{x}}_k^-)
   $$

    Note 1: the residual $r_k$ s also called the **innovation**.

    Note 2: $\hat{x}_k^- \neq x_k$. First incorporates prediction error, second is the hidden state we want to estimate. Therefore $r_k \neq v_k$

2. Compute Kalman gain:

   $$S_k=H_kP_k^-H_k^T+R_k,$$
   $$K_k=P_k^-H_k^TS_k^{-1}$$

   Implement the gain using a linear solve rather than explicitly inverting $S_k$.

3. Correct the state:

   $$
   \hat{\mathbf{x}}_k^+=\hat{\mathbf{x}}_k^-+K_k\mathbf{r}_k
   $$

4. Correct the state estimate error covariance matrix:

   $$
   P_k^+=(I-K_kH_k)P_k^-
   $$

    Note: production implementations often use a numerically more robust form, such as the Joseph covariance update (this form guarantees the updated covariance remains symmetric positive semi-definite):
   $$
   P_k^+=(I-K_kH_k)P_k^-(I-K_kH_k)^T+K_kR_kK_k^T
   $$

## 3. Integration with a VO system: EKF-based VIO

### Model and notation
State:

$$\mathbf{x}=(\mathbf{p},\mathbf{v},\mathbf{q},\mathbf{b}_g,\mathbf{b}_a)$$

where:
- $p \in \mathbb{R}^3$ - position.
- $v \in \mathbb{R}^3$ - velocity.
- $q$ - **orientation quaternion**.
- $b_g \in \mathbb{R}^3$ - gyroscope bias
- $b_a \in \mathbb{R}^3$ - accelerometer bias

This is a conceptual state, not a complete description of every VIO implementation. A typical error-state representation has 15 error variables: three for each position.
The **orientation quaternion** itself uses four numbers to represent rotation, but only three independent rotational degrees of freedom.

Its local error state has $15$ components (three per quantity), with $P\in\mathbb{R}^{15\times15}$ before adding landmarks or past poses.

### IMU input → prediction

For each time interval $\Delta t$ use IMU to propagate state. Remove biases and account for gravity:

$$
\boldsymbol{\omega}=\boldsymbol{\omega}_m-\mathbf{b}_g,
$$
$$\mathbf{a}_W=R_{WB}(\mathbf{a}_m-\mathbf{b}_a)+\mathbf{g}_W$$

The system then integrates the motion:
$$
\mathbf{p}_{k+1}\approx\mathbf{p}_k+\mathbf{v}_k\Delta t+\tfrac12\mathbf{a}_W\Delta t^2,
$$

$$\mathbf{v}_{k+1}\approx\mathbf{v}_k+\mathbf{a}_W\Delta t$$

Note 1: $R_{WB}$ rotation matrix, maps body coordinates into the world frame.

Note 2: The IMU isn't perfectly accurate. As the system integrates noisy measurements, uncertainty generally grows. The filter propagates that uncertainty using $F$ and $Q$.

Note 3: These equations are simplified discretizations; practical systems account for changing orientation during an interval and propagate the covariance using the linearized inertial dynamics.

### Camera frame → visual correction

There are two choices for integration of VO and IMU:

| Approach | EKF measurement |
|---|---|
| Loosely coupled | VO pose or relative-pose estimate, with its covariance and consistent frame/scale conventions |
| Tightly coupled | Tracked image coordinates, using reprojection residuals directly |

The system predicts where the observed 3D feature should appear in the current image, given its estimated pose and position. It compares that prediction with the actual detected image coordinate.

The measurement residual is:

$$r = z - \hat{z}$$

For a feature with image coordinates $(u,v)$:
$$z = \begin{bmatrix}u\\v\end{bmatrix}$$

The predicted coordinates come from projecting the feature into the camera:

$$\hat{\mathbf{z}}_i=\pi\!\left(R_{CW}(\mathbf{L}_{W}-\mathbf{p}_{W})\right)$$

where:
- $L_W$ - is the feature's 3D position in the world.
- $p_W, R_CW$ - describe the camera pose, with rotation conventions chosen consistently.
- $\pi$ - the camera projection function, including perspective division and, where appropriate, camera intrinsics.

The measurement Jacobian $H$ describes how this predicted image location changes when the state changes. Its columns can capture sensitivity to position, orientation, and potentially other estimated variables.

The EKF uses $H$, the residual, and the covariances to correct the state.

In a tightly coupled VIO filter, you can instead use feature observations to correct the entire state estimate through the measurement Jacobian and covariance. The correction can affect not only the current pose but also velocity, biases, and other correlated variables.